# 05 - Scenario Analyse

## Doel
Bepalen of ik mijn doel van EUR 100.000 in goud zal bereiken in 3 jaar,
op basis van 3 scenario's gebaseerd op historische rendementen.

## Hoe groeivoeten worden bepaald (data-gedreven)
1. Historische goudrendementen over 6 jaar (2020-2026) berekenen
2. Percentielen uit de data:
   - **Bullish**: 6-jaar CAGR (~+18%)
   - **Neutraal**: Mediaan (~+7%)
   - **Bearish**: 25e percentiel (~+5%)
3. Aanpassing voor huidig macro-klimaat (centrale banken, geopolitiek)

In [ ]:
# Cel 1: Inladen van bibliotheken en instellen van paden

import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.utils import (
    load_portfolio, calculate_pure_gold_weight,
    filter_bars, calculate_portfolio_value, goal_progress,
    next_purchase_date, format_eur, format_gram
)
from src.data_loader import load_raw

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
sns.set_palette('husl')

TROY_OZ_TO_GRAM = 31.1035
PROC_DIR = Path('../data/processed')
print('Setup complete.')

In [ ]:
# Cel 2: Aanpassing voor huidig macro-klimaat (centrale banken, geopolitiek)

# Laad data
gold = pd.read_csv(PROC_DIR / 'gold_with_macro.csv', index_col=0, parse_dates=True)
gold.index = pd.to_datetime(gold.index)

portfolio = pd.read_csv(PROC_DIR / 'portfolio_cleaned.csv')
portfolio['datum_aankoop'] = pd.to_datetime(portfolio['datum_aankoop'])
bars = pd.read_csv(PROC_DIR / 'bars_with_spot.csv')
bars['datum_aankoop'] = pd.to_datetime(bars['datum_aankoop'])

# Huidige status
stats = calculate_portfolio_value(portfolio, gold)
print(f'Huidige waarde portefeuille: {format_eur(stats["total_portfolio_value_eur"])}')
print(f'Huidige geinvesteerd: {format_eur(stats["total_invested_eur"])}')

## 1. Historische Rendementen Berekenen

In [ ]:
# Cel 3: Historische Rendementen Berekenen

# Maandelijkse rendementen
col = 'gold_eur_gram' if 'gold_eur_gram' in gold.columns else 'gold_usd_oz'
monthly_prices = gold[col].resample('ME').last().dropna()
monthly_returns = monthly_prices.pct_change().dropna()

# Jaarlijkse rendementen
yearly_prices = gold[col].resample('YE').last().dropna()
yearly_returns = yearly_prices.pct_change().dropna()

print('--- Jaarlijkse Rendementen ---')
for date, ret in yearly_returns.items():
    print(f'  {date.year}: {ret:+.2%}')

print(f'\n--- Statistieken (10jr) ---')
print(f'Gem. jaarlijks rendement: {yearly_returns.mean():.2%}')
print(f'Std jaarlijks rendement:  {yearly_returns.std():.2%}')
print(f'Mediaan:                  {yearly_returns.median():.2%}')
print(f'25e percentiel:           {yearly_returns.quantile(0.25):.2%}')
print(f'75e percentiel:           {yearly_returns.quantile(0.75):.2%}')

In [ ]:
# Cel 4: Historische Rendementen Berekenen

# Visuele inspectie van historische rendementen
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Maandelijkse rendementen
axes[0].bar(monthly_returns.index, monthly_returns.values, width=20,
            color=['green' if r > 0 else 'red' for r in monthly_returns.values], alpha=0.7)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].set_title('Maandelijkse Goudrendementen')
axes[0].set_ylabel('Rendement')

# Jaarlijkse rendementen
colors = ['green' if r > 0 else 'red' for r in yearly_returns.values]
axes[1].bar(range(len(yearly_returns)), yearly_returns.values, color=colors, alpha=0.7)
axes[1].set_xticks(range(len(yearly_returns)))
axes[1].set_xticklabels([str(d.year) for d in yearly_returns.index])
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_title('Jaarlijkse Goudrendementen')
axes[1].set_ylabel('Rendement')

plt.tight_layout()
plt.show()

## 2. Scenario's Definieren

Gebaseerd op historische percentielen + huidig macro-klimaat.

In [ ]:
# Cel 5: Gebaseerd op historische percentielen + huidig macro-klimaat.

# Scenario parameters (data-gedreven)
p25 = yearly_returns.quantile(0.25)
p50 = yearly_returns.median()
p75 = yearly_returns.quantile(0.75)

# Aanpassing voor huidig klimaat (2025-2026)
# Centrale banken kopen record hoeveelheden goud
# Geopolitieke onzekerheid hoog
# Inflatie nog steeds verhoogd
climate_adjustment = 0.01  # +1% voor bullish vanwege sterk centraal bank demand

scenarios = {
    'Bullish (+12%)': {
        'annual_return': p75 + climate_adjustment,
        'description': '6-jaar CAGR 2020-2026',
        'color': 'green'
    },
    'Neutraal (+5%)': {
        'annual_return': p50,
        'description': 'Mediaan historisch rendement',
        'color': 'gold'
    },
    'Bearish (+1%)': {
        'annual_return': p25,
        'description': '25e percentiel (onderste kwartiel)',
        'color': 'red'
    }
}

print('--- Scenario Definitie ---')
for name, s in scenarios.items():
    print(f'{name}: {s["annual_return"]:.1%} per jaar')
    print(f'  Bron: {s["description"]}')
    print()

## 3. Projektie: Portefeuille Waarde over 3 Jaar

In [ ]:
# Cel 6: Projektie: Portefeuille Waarde over 3 Jaar

# Projectie parameters
huidige_waarde = stats['total_portfolio_value_eur']
huidige_spot   = gold['gold_eur_gram'].iloc[-1]
gem_premium    = 0.0371   # Gemiddelde dealerpremium op basis van historische aankopen
bar_gram       = 10       # Gram per nieuwe aankoop
interval_mnd   = 2        # Elke 2 maanden 1 staaf kopen
aantal_jaren   = 3
aantal_maanden = aantal_jaren * 12

print('--- Projektie Parameters ---')
print(f'Huidige waarde:          {format_eur(huidige_waarde)}')
print(f'Huidige spot EUR/gram:   {format_eur(huidige_spot)}')
print(f'Gem. dealer premium:     {gem_premium:.2%}')
print(f'Aankoop: {bar_gram}g staaf elke {interval_mnd} maanden')
print(f'Projektie horizon:       {aantal_jaren} jaar ({aantal_maanden} maanden)')
print()

dates = pd.date_range(start=gold.index[-1], periods=aantal_maanden + 1, freq='ME')
projections      = {}   # portfolio EUR waarde per maand
invested_extra   = {}   # cumulatief extra geïnvesteerd

for name, scenario in scenarios.items():
    monthly_rate = (1 + scenario['annual_return']) ** (1/12) - 1
    spot         = huidige_spot
    waarde       = huidige_waarde
    extra_inv    = 0
    values       = [waarde]
    inv_values   = [0]

    for month in range(1, aantal_maanden + 1):
        # Spot prijs groeit mee met het scenario
        spot = spot * (1 + monthly_rate)

        # Elke 2 maanden een nieuwe 10g staaf kopen
        if month % interval_mnd == 0:
            aankoopprijs = bar_gram * spot * (1 + gem_premium)
            # Voeg gram toe: aankoopprijs / spot = bar_gram gram
            waarde      += bar_gram * spot   # marktwaarde van de nieuwe staaf
            extra_inv   += aankoopprijs

        # Groei op hele portefeuille
        waarde = waarde * (1 + monthly_rate)
        values.append(waarde)
        inv_values.append(extra_inv)

    projections[name]    = values
    invested_extra[name] = inv_values
    print(f'{name}: {format_eur(huidige_waarde)} -> {format_eur(values[-1])} '
          f'(+{format_eur(inv_values[-1])} extra geïnvesteerd)')


In [ ]:
# Cel 7: Projektie: Portefeuille Waarde over 3 Jaar

# Visuele projektie (inclusief nieuwe aankopen)
fig, ax = plt.subplots(figsize=(14, 7))

# Doellijn
ax.axhline(100_000, color='black', linestyle='-', linewidth=2, label='Doel: EUR 100.000')

for name, values in projections.items():
    color = scenarios[name]['color']
    ax.plot(dates, values, color=color, linewidth=2,
            label=f'{name} ({scenarios[name]["annual_return"]:.0%}/jr)')
    ax.fill_between(dates, values, alpha=0.08, color=color)
    ax.annotate(f'{format_eur(values[-1])}',
                xy=(dates[-1], values[-1]),
                xytext=(10, 0), textcoords='offset points',
                fontsize=10, color=color, fontweight='bold')

# Toon aankoopmomenten (bestaand)
for _, row in bars.iterrows():
    ax.axvline(pd.Timestamp(row['datum_aankoop']), color='gray', linestyle=':', alpha=0.3)

# Toon geplande toekomstige aankopen (elke 2 maanden)
laatste_aankoop = pd.Timestamp(bars['datum_aankoop'].max())
for i in range(1, 19):
    toekomstig = laatste_aankoop + pd.DateOffset(months=i*2)
    ax.axvline(toekomstig, color='blue', linestyle=':', alpha=0.15)

ax.set_title(f'Portefeuille Projektie: 3 Scenario (incl. nieuwe aankopen elke 2 mnd)\n'
             f'Huidig: {format_eur(huidige_waarde)} | Doel: EUR 100.000', fontsize=13)
ax.set_ylabel('Portefeuille Waarde (EUR)')
ax.set_xlabel('Datum')
ax.legend(loc='upper left')
ax.set_ylim(0, max(max(v) for v in projections.values()) * 1.15)
plt.tight_layout()
plt.show()


## 4. Doel Analyse per Scenario

In [ ]:
# Cel 8: Doel Analyse per Scenario

print('=' * 60)
print('   DOEL ANALYSE: EUR 100.000 IN 3 JAAR')
print('='  * 60)
print(f'Huidige waarde:  {format_eur(huidige_waarde)}')
print(f'Tekort nu:       {format_eur(100_000 - huidige_waarde)}')
print()

doel_bereikt = {}
for name, values in projections.items():
    eind       = values[-1]
    bereikt    = eind >= 100_000
    doel_bereikt[name] = bereikt
    extra      = invested_extra[name][-1]

    maand_bereikt = next((i for i, v in enumerate(values) if v >= 100_000), None)

    print(f'{name}:')
    print(f'  Eindwaarde (3jr):    {format_eur(eind)}')
    print(f'  Extra geïnvesteerd:  {format_eur(extra)}')
    print(f'  Doel bereikt:        {"Ja" if bereikt else "Nee"}')
    if maand_bereikt:
        print(f'  Wanneer:             Maand {maand_bereikt} ({dates[maand_bereikt].strftime("%B %Y")})')
    else:
        print(f'  Tekort na 3 jaar:    {format_eur(100_000 - eind)}')
    print()

print('=' * 60)


## 5. Gevoeligheidsanalyse

Wat als de goudprijs sneller/langzamer stijgt?

In [ ]:
# Cel 9: Wat als de goudprijs sneller/langzamer stijgt?

# Gevoeligheid: hoeveel % per jaar is nodig om EUR 100K te bereiken?
doel = 100_000
jaren = 3

# Benodigd rendement
benodigd = (doel / huidige_waarde) ** (1/jaren) - 1

print(f'--- Gevoeligheidsanalyse ---')
print(f'Om {format_eur(doel)} te bereiken vanuit {format_eur(huidige_waarde)}:')
print(f'Benodigd jaarlijks rendement: {benodigd:.1%}')
print()

# Verschillende scenario's
test_rendementen = [0.00, 0.03, 0.05, 0.08, 0.10, 0.12, 0.15, benodigd]
print('Rendement  | Eindwaarde  | Doel bereikt?')
print('-' * 50)
for r in test_rendementen:
    factor = (1 + r) ** jaren
    eind = huidige_waarde * factor
    bereikt = 'Ja' if eind >= doel else 'Nee'
    marker = ' <-- BENODIGD' if abs(r - benodigd) < 0.001 else ''
    print(f'{r:>8.1%}   | {format_eur(eind):>12} | {bereikt}{marker}')

In [ ]:
# Cel 10: Wat als de goudprijs sneller/langzamer stijgt?

# Gevoeligheidsgrafiek
fig, ax = plt.subplots(figsize=(14, 6))

rendementen_range = np.linspace(-0.05, 0.20, 100)
eind_waarden = [huidige_waarde * (1 + r) ** jaren for r in rendementen_range]

ax.plot(rendementen_range * 100, eind_waarden, color='steelblue', linewidth=2)
ax.axhline(100_000, color='red', linestyle='--', linewidth=2, label='Doel: EUR 100.000')
ax.axvline(benodigd * 100, color='green', linestyle='--', linewidth=1.5, 
           label=f'Benodigd: {benodigd:.1%}')
ax.fill_between(rendementen_range * 100, eind_waarden, 100_000,
               where=[e >= 100_000 for e in eind_waarden],
               alpha=0.2, color='green', label='Doel bereikt')
ax.fill_between(rendementen_range * 100, eind_waarden, 100_000,
               where=[e < 100_000 for e in eind_waarden],
               alpha=0.2, color='red', label='Doel niet bereikt')

ax.set_title(f'Gevoeligheidsanalyse: Benodigd Rendement voor EUR 100.000\n(Over {jaren} jaar vanuit {format_eur(huidige_waarde)})', fontsize=14)
ax.set_xlabel('Jaarlijks Rendement (%)')
ax.set_ylabel('Eindwaarde (EUR)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Aanbeveling & Conclusie

In [ ]:
# Cel 11: Aanbeveling & Conclusie

print('=' * 60)
print('   AANBEVELING')
print('=' * 60)
print()
print('Op basis van de analyse (incl. nieuwe aankopen elke 2 maanden):')
print()

neutraal_key = next((k for k in doel_bereikt if 'Neutraal' in k), None)
bullish_key  = next((k for k in doel_bereikt if 'Bullish' in k), None)

if neutraal_key and doel_bereikt[neutraal_key]:
    print('  [OK] Neutraal scenario: Doel van EUR 100.000 wordt bereikt')
elif bullish_key and doel_bereikt[bullish_key]:
    print('  [WARN] Alleen het bullish scenario haalt het doel')
else:
    print('  [RISK] Doel van EUR 100.000 wordt in geen enkel scenario bereikt')
    print('         binnen 3 jaar, zelfs met nieuwe aankopen elke 2 maanden.')
    print()
    # Bereken hoeveel gram per aankoop nodig zou zijn
    print('  Alternatieve opties:')
    print('  - Horizon verlengen (bijv. 4-5 jaar)')
    print('  - Grotere staven kopen (bijv. 20g i.p.v. 10g)')
    print('  - Hogere aankoopfrequentie (elke maand i.p.v. elke 2 maanden)')

print()
print('Aanbevelingen:')
print('  1. Monitor de portefeuille elke 2 maanden')
print('  2. Als de feitelijke groei onder het bearish scenario zit,')
print('     overweeg het budget te verhogen of het doel aan te passen')
print('  3. De modelvoorspelling (NB 04) geeft een richtwaarde;')
print('     gebruik deze samen met de scenario-analyse')
print('  4. Goud is een veilige haven maar geen garantie')
print()
print('Disclaimer: Dit is educatieve analyse, geen financieel advies.')
print('=' * 60)


---
## Samenvatting

| Scenario | Jaarlijks Rendement | Eindwaarde (3jr) | Doel Bereikt? |
|----------|-------------------|------------------|---------------|
| Bullish | TBD | TBD | TBD |
| Neutraal | TBD | TBD | TBD |
| Bearish | TBD | TBD | TBD |

**Benodigd rendement om EUR 100K te bereiken:** TBD%

